_______
### 0. Preamble
- This notebook is to simply document my own notes on utilising the firestore database that I have written down in the process of developing my own bot

_______
### 1. Imports

In [1]:
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime, timedelta

_______
### 2. Basic functions

In [3]:
def initialise_creds():
    if not firebase_admin._apps:
        cred = credentials.Certificate("firebase_service_account.json")
        firebase_admin.initialize_app(cred)

- this method only needs to be called once to validate the credentials required so that subsequent calls can access the firebase client

In [ ]:
db = firestore.client() # to access the database 

#### 2.1 CREATE (POST)
- **CREATE** as a form of manipulating the database is done with **POST** type of HTTP webhooks

In [ ]:
def POST(payload: dict, id : int, table : str, inner_table : str):
    _, doc_ref = db.collection(table).document(id).collection(inner_table).add(payload)
    return doc_ref.id

- `db.collection(table)` indicates the table that is being accessed
- chaining `.document(id)` indicates the unique id of the entry created
- chaining `.collection(inner_table).add(payload)` then updates the entry with the information in the payload
- `doc_ref.id` is the returned auto generated id

In [ ]:
def POST2(payload: dict, id : int, table: str):
    _, doc_ref = db.collection(table).document(id).set(payload)
    return doc_ref.id

- `.set()` can be used instead when the ID is meaningful, like the Telegram user ID

#### 2.2 READ (GET)
- **READ**ing data from the database is done with **READ** type of HTTP webhooks

In [ ]:
def READ(id: int, table: str):
    doc = db.collection(table).get(id)
    return doc.to_dict(), doc.id

- `db.collection(table)` indicates the table that is being accessed
- chaining `.get(id)` obtains the entry based on the identifying id 
- `.to_dict()` converts the entry into a dictionary
- `doc.id` returns the document id

In [ ]:
def READ_ALL(id: int, table: str, inner_table : str):
    docs = db.collection(table).document(id).collection(inner_table).stream()
    return docs

- `db.collection(table)` indicates the table that is being accessed
- chaining `.document(id)` obtains the entry based on the identifying id
- chaining `.collection(inner_table)` obtains all entries inside the document with the corresponding id

#### 2.3 UPDATE (PUT)
- **UPDATE**ing data from the database is done with **PUT** type of HTTP webhooks

In [ ]:
def UPDATE(updated_info: dict, table : str, id : int, inner_table : str, inner_id : int):
    event_ref = db.collection(table).document(id).collection(inner_table).document(inner_id)
    event_ref.update(updated_info)

- `event_ref = db.collection(table).document(id).collection(inner_table).document(inner_id)` obtains the exact entry that needs to be updated
- chaining `.update(updated_info)` only touches the fields you specify, everything else unchanged

In [ ]:
def UPDATE(updated_info: dict, table : str, id : int, inner_table : str, inner_id : int):
    event_ref = db.collection(table).document(id).collection(inner_table).document(inner_id)
    event_ref.set(updated_info, merge=True)

- `set(updated_info, merge=True)` acts the same as `update()` but creates the document if it doesn't exist

In [ ]:
def UPDATE(updated_info: dict, table : str, id : int, inner_table : str, inner_id : int):
    event_ref = db.collection(table).document(id).collection(inner_table).document(inner_id)
    event_ref.set(updated_info)

- `set(updated_info)` without merge will overwrites the ENTIRE document, all other fields are wiped

#### 2.4 DELETE (DELETE)
- **DELETE**ing data from the database is done with **DELETE** type of HTTP webhooks

In [ ]:
def delete(table : str, id : int, inner_table : str, inner_id : int):
    db.collection(table).document(id).collection(inner_table).document(inner_id).delete()

- just chain `.delete()` at the back after finding the document that needs to be deleted